# 📊 Feature Exploration

This notebook demonstrates the feature extraction pipeline:
- MFCC computation
- Mel-spectrogram visualisation
- Temporal feature analysis
- Feature distributions across trigger classes and accents

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.features.extractor import FeatureExtractor
from src.features.audio_utils import AudioUtils
from src.pipeline.data_generator import DataGenerator

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

extractor = FeatureExtractor(config['features'], config['audio']['sample_rate'])
print(f'Feature dimension: {extractor.feature_dim}')

## Synthesize Demo Waveforms

In [ ]:
sr = config['audio']['sample_rate']
duration = config['audio']['duration_sec']
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# 4 trigger classes
signals = {
    'Wake-word A (300 Hz)': 0.7 * np.sin(2 * np.pi * 300 * t) * np.exp(-t),
    'Wake-word B (800 Hz)': 0.9 * np.sin(2 * np.pi * 800 * t) * np.exp(-t * 1.5),
    'Command (1500 Hz)':   0.6 * np.sin(2 * np.pi * 1500 * t) * np.exp(-t * 0.8),
    'Noise (3000 Hz)':     0.5 * np.sin(2 * np.pi * 3000 * t) * np.exp(-t * 2.0),
}

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for ax, (name, sig) in zip(axes, signals.items()):
    ax.plot(t[:1600], sig[:1600], linewidth=0.5)
    ax.set_ylabel(name, fontsize=9)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Synthetic Trigger Waveforms', fontsize=13)
fig.tight_layout()
plt.show()

## MFCC Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (name, sig) in zip(axes.flatten(), signals.items()):
    mfcc = extractor.extract_mfcc(sig.astype(np.float32))
    ax.imshow(mfcc, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(name, fontsize=10)
    ax.set_ylabel('MFCC coeff')
fig.suptitle('MFCC Features', fontsize=13)
fig.tight_layout()
plt.show()

## Mel-Spectrogram Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (name, sig) in zip(axes.flatten(), signals.items()):
    mel = extractor.extract_mel_spectrogram(sig.astype(np.float32))
    ax.imshow(mel, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(name, fontsize=10)
    ax.set_ylabel('Mel band')
fig.suptitle('Log-Mel Spectrograms', fontsize=13)
fig.tight_layout()
plt.show()

## Temporal Feature Comparison

In [ ]:
import pandas as pd

temporal_data = {}
for name, sig in signals.items():
    feats = extractor.extract_temporal(sig.astype(np.float32))
    temporal_data[name] = feats

feat_labels = ['Zero-Crossing Rate', 'RMS Energy', 'Spectral Centroid', 'Spectral Rolloff']
df = pd.DataFrame(temporal_data, index=feat_labels[:len(feats)])
df.T.plot(kind='bar', figsize=(12, 5), colormap='Set2')
plt.title('Temporal Features by Trigger Class')
plt.ylabel('Value')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Accent Perturbation Effect

In [ ]:
base_sig = list(signals.values())[0].astype(np.float32)
accents = config['data']['accents']

fig, axes = plt.subplots(len(accents), 1, figsize=(14, 8), sharex=True)
for ax, accent in zip(axes, accents):
    perturbed = AudioUtils.apply_accent_perturbation(base_sig, accent, sr)
    ax.plot(t[:1600], perturbed[:1600], linewidth=0.5)
    ax.set_ylabel(accent, fontsize=9)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Accent Perturbation on Wake-word A', fontsize=13)
fig.tight_layout()
plt.show()

## Feature Distribution (Simulated Data)

In [ ]:
gen = DataGenerator(config)
X, y, acc_ids, noise_ids = gen.generate_fast(num_samples=5000)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for d, ax in enumerate(axes.flatten()):
    for cls in range(config['data']['num_classes']):
        mask = y == cls
        ax.hist(X[mask, d], bins=30, alpha=0.5, label=f'Class {cls}', density=True)
    ax.set_title(f'Feature {d}', fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle('Feature Distributions by Class', fontsize=13)
fig.tight_layout()
plt.show()